In [14]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [15]:
# read in all the words

words = open('../data/names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [16]:
len(words)

32033

In [17]:
# build the vocabulary of characters and mapping to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [26]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words:
  
  #print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    #print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)

In [27]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [22]:
C = torch.randn((27, 2))

In [42]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [43]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [50]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)

In [51]:
h

tensor([[ 0.9694,  0.5147,  0.3220,  ..., -0.9991,  0.3066,  0.9152],
        [ 0.9603, -0.2450,  0.8824,  ..., -0.9980,  0.1321,  0.6562],
        [ 0.7692, -1.0000, -0.9911,  ..., -0.9918, -0.9801, -0.9694],
        ...,
        [ 0.9553, -0.9602,  0.2659,  ..., -0.5442, -0.9924, -0.9595],
        [-0.0741, -1.0000, -0.9962,  ..., -1.0000, -0.9317, -0.9952],
        [-0.3510,  0.6035, -1.0000,  ...,  0.8225,  0.5550, -0.9973]])

In [52]:
h.shape

torch.Size([32, 100])

In [53]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [54]:
logits = h @ W2 + b2

In [55]:
logits.shape

torch.Size([32, 27])

In [56]:
counts  = logits.exp()

In [60]:
prob = counts / counts.sum(1, keepdim=True)

In [61]:
prob.shape

torch.Size([32, 27])

In [64]:
loss = -prob[torch.arange(32), Y].log().mean()
loss

tensor(17.0528)

In [ ]:
# Made more respectable

In [28]:
X.shape, Y.shape # dataset

(torch.Size([228146, 3]), torch.Size([228146]))

In [52]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [53]:
sum(p.nelement() for p in parameters) # number of parameters in total

3481

In [54]:
for p in parameters:
    p.requires_grad = True

In [83]:
for _ in range(1000):
    #minibatch construct
    ix = torch.randint(0, X.shape[0], (32, ))
    #forward pass
    emb = C[X[ix]]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y[ix])
    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    #update
    for p in parameters:
        p.data += -0.1 * p.grad

print(loss.item())


2.641789197921753


In [84]:
emb = C[X]
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Y)
loss

tensor(2.5375, grad_fn=<NllLossBackward0>)